### Set up

Issues: Conversion of I to Ziv -- inconsistent in notebook cells
    Figure out automatic conversion to 1st person for nie scenario processing

In [1]:
import json # !pip
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
from pathlib import Path
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)

<module 'src.translate_to_vis' from '/Users/rylenc/Dropbox/2025_moral_scenario_annotation/code/rylen/graph_extract/src/translate_to_vis.py'>

In [2]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [3]:
# set main paths
'''
CUR_DIR = os.path.dirname(os.path.abspath(__name__))
SCENARIO_DIR = CUR_DIR+'/scenarios/'
DATA_DIR_HUMAN = CUR_DIR+'/data/human_annotation/'
OUTPUT_DIR = CUR_DIR+'/annotated_outputs/'
'''
CUR_DIR = Path().resolve()
print(f"current_path: {CUR_DIR}")

SCENARIO_DIR = CUR_DIR / "scenarios"
DATA_DIR_HUMAN = CUR_DIR / "data" / "human_annotation"
OUTPUT_DIR = CUR_DIR / "annotated_outputs"

current_path: /Users/rylenc/Dropbox/2025_moral_scenario_annotation/code/rylen/graph_extract


## Look at Scenario

In [60]:
#set scenario file filename
FILENAME = 'nie_scenarios.json'

#select scenario and action choice
SCENARIO_ID = 3  # Note this is the index from the scenarios JSON file, not the ID
ACT_ID = '2'

#read in the scenario
scenario_json = utils.open_scenario(SCENARIO_DIR, FILENAME, SCENARIO_ID, ACT_ID)


Scenario Text: 


 On the test ground of a modern railroad property an unmanned speed-train that normally can be
remote-controlled got out of control due to a technical defect. This speed-train is heading towards
three railroad workers that are maintaining the tracks. Since these workers are wearing a novel
hearing protection, they would not notice the speed-train on time and hence would be run over by it.
I, an employee of the rail track control center, recognize the upcoming accident. However, it is not
possible to stop the train on time anymore. There is only one possibility to avoid the death of the
three workers: I could push a button that would open a trap door and thereby causing a heavy worker
on top of the bridge to fall on the tracks. The speed-train would collide with the heavy worker and
be stopped before it reaches the three workers. The heavy worker would lose his life due to the
collision. 




#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Score the action in moral value

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

#### Set up

In [61]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
this_act_I = "I decide to " + this_act
this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)
print('\n\nAction choice:') 
print(this_act_Ziv)
print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)
print("\n\nScenario:")
print(this_scenario_Ziv)
print(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act_I}





Action choice:
Ziv decides not to push the button.
I decide to not push the button


Scenario:
On the test ground of a modern railroad property, an unmanned speed-train that normally can be remote-controlled got out of control due to a technical defect. This speed-train is heading towards three railroad workers who are maintaining the tracks. Since these workers are wearing a novel hearing protection, they would not notice the speed-train in time and hence would be run over by it. Ziv, an employee of the rail track control center, recognizes the upcoming accident. However, it is not possible for Ziv to stop the train on time anymore. There is only one possibility to avoid the death of the three workers: Ziv could push a button that would open a trap door and thereby cause a heavy worker on top of the bridge to fall on the tracks. The speed-train would collide with the heavy worker and be stopped before it reaches the three workers. The heavy worker would lose his life due to the coll

In [62]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')

Graph g initialized and reset.


#### Step 0. Get entities

In [63]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = annotate_scenario.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]

#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list



Identified these entities: 

I
three railroad workers
heavy worker on top of the bridge


#### Step 1. Deontology / Action Value Scores

In [64]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to rate the moral goodness or wrongness of the action with no context
processed_values  = annotate_scenario.process_value_simple(this_act,this_act_I,g) 

print(processed_values)

    

You are an expert judge of the moral virtues of different actions. Moral actions are those which humans consider virtuous, that consider others' wellbeing and happiness, and that are guided by principles of ethics. You will be given a sentence describing an action. You must rate thus action in terms of its moral virtue. Respond with a rating from -100 to 100 for this action, where -100 is the most morally wrong action you can possibly think of, and 100 is the most morally virtuous action you can possibly think of. 0 indicates a morally neutral action. Please return a json object with key:value pair of "score": rating. Please rate this action: I decide to not push the button
{'score': 0}


#### Step 2. Outcome Likelihoods

In [65]:
#Step 2. Outcomes

processed_events = annotate_scenario.process_outcomes(this_scenario, this_act)
events_I= processed_events[1]
events_Ziv= processed_events[0]
print("\n".join(events_I))         
scenario_dict["outcomes"]= events_I

The speed-train continues on its current path
The speed-train collides with the three railroad workers
The heavy worker on top of the bridge remains unharmed
I do not intervene in the situation.


#### Step 3. Outcome Utilities

In [66]:
#Step 3. Outcome utilities

impacts_list = annotate_scenario.process_impacts(this_scenario_Ziv, this_act, this_act_Ziv, events_Ziv, events_I, beings_Ziv, g) 



Processing impacts of event: The speed-train continues on its current path
Scored impacts for these beings:
['Ziv', 'three railroad workers', 'heavy worker on top of the bridge']
Scored values:
[-80, -100, 0]

Processing impacts of event: The speed-train collides with the three railroad workers
Found unknown being three_railroad_workers at index 1
Fixed beings list:
['Ziv', 'three railroad workers', 'heavy_worker_on_top_of_the_bridge']
Found unknown being heavy_worker_on_top_of_the_bridge at index 2
Fixed beings list:
['Ziv', 'three railroad workers', 'heavy worker on top of the bridge']
Scored impacts for these beings:
['Ziv', 'three railroad workers', 'heavy worker on top of the bridge']
Scored values:
[-80, -100, 0]

Processing impacts of event: The heavy worker on top of the bridge remains unharmed
Scored impacts for these beings:
['Ziv', 'three railroad workers', 'heavy worker on top of the bridge']
Scored values:
[0, -100, 100]

Processing impacts of event: I do not intervene in

#### Step 4. Cause / Intend / Know Links

In [67]:
#Step 4. causal / intentional / knowledge links -- run on currently generated event/outcome list
output_links = annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act_Ziv,g)    


Processing event: The speed-train continues on its current path
{'cause': 'no', 'intend': 'no', 'know': 'yes'}
CKI links for I
C-I-K+

Processing event: The speed-train collides with the three railroad workers
{'cause': 'no', 'intend': 'no', 'know': 'yes'}
CKI links for I
C-I-K+

Processing event: The heavy worker on top of the bridge remains unharmed
{'cause': 'no', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C-I+K+

Processing event: I do not intervene in the situation.
{'cause': 'no', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C-I+K+


#### Step 5. Write out the results

In [ ]:
#optional -- write out the results 

this_output_filename = f"nie_scenarios_{scenario_json["id"]}_choice_{ACT_ID}.json"
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(this_output_filename, g_print)
print('\n\n')


translate_to_vis.main(this_output_filename)




Writing to file: nie_scenarios_3_choice_2.json



nie_scenarios_3_choice_2.json
